In [1]:
# =====================================================================
# CELL 1 — Install and import
# =====================================================================
# VS Code / local:  pip install pyswarms imbalanced-learn scikit-learn pandas matplotlib
# Colab:            uncomment the line below
# !pip install pyswarms imbalanced-learn -q

import os, time, json, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pyswarms as ps

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FIGDIR = 'figures'
os.makedirs(FIGDIR, exist_ok=True)

# Every figure is written at the exact pixel size of the image it replaces
# in the Word document, so it drops straight in without re-scaling.
FIGSPEC = {
    'fig_4_1_class_distribution':  (1317, 710),
    'fig_4_2_cm_lr':               (1347, 1176),
    'fig_4_3_convergence':         (1318, 784),
    'fig_4_4_feature_selection':   (1318, 560),
    'fig_4_5_cm_pso_lr':           (1347, 1176),
    'fig_4_6_performance':         (1307, 647),
    'fig_4_7_computation_time':    (1307, 647),
}


def new_fig(name):
    """Open a figure sized to match the document image it replaces."""
    w, h = FIGSPEC[name]
    return plt.subplots(figsize=(w / 100, h / 100), dpi=100)


def save_fig(fig, name):
    fig.savefig(f'{FIGDIR}/{name}.png', dpi=100)
    plt.close(fig)
    print(f'  saved {FIGDIR}/{name}.png')


In [2]:
# =====================================================================
# CELL 2 — Load the dataset
# =====================================================================
# VS Code / local:  put weatherAUS.csv beside this notebook
# Colab:            uncomment to upload it
# from google.colab import files; files.upload()
df = pd.read_csv('weatherAUS.csv')
print('Raw shape:', df.shape)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'weatherAUS.csv'

In [ ]:
# =====================================================================
# CELL 3 — Preprocessing
#
# CHANGED FROM THE ORIGINAL: imputation and scaling are NO LONGER done
# here. Only operations that cannot leak — dropping rows with a missing
# target, deriving cyclic month features, dropping Location, and mapping
# binary categoricals — happen outside the cross-validation loop.
# Imputation and scaling move into the pipeline (Cell 4) so they are
# fitted on training folds only.
# =====================================================================
df = df.dropna(subset=['RainTomorrow']).copy()

# Cyclic month encoding (derived from the row's own Date - cannot leak)
df['Date'] = pd.to_datetime(df['Date'])
df['Month_sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)
df = df.drop(columns=['Date'])

# Location dropped: too many categories to encode usefully here
df = df.drop(columns=['Location'])

# Binary categoricals: a fixed, data-independent mapping
df['RainToday'] = df['RainToday'].map({'Yes': 1, 'No': 0})
df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})

# Wind directions: label encoding is a fixed lexicographic mapping over the
# 16 compass points, so fitting it on all rows carries no target information.
wind_cols = ['WindGustDir', 'WindDir9am', 'WindDir3pm']
for col in wind_cols:
    df[col] = df[col].fillna('Unknown')
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X_df = df.drop(columns=['RainTomorrow'])
X = X_df.values                      # NOTE: unscaled, still contains NaNs
y = df['RainTomorrow'].values
feature_names = X_df.columns.tolist()

print('After preprocessing:', X.shape)
print('Total features:', len(feature_names))
print('Class distribution:', np.bincount(y))
print('Remaining NaNs (handled inside the pipeline):', int(np.isnan(X).sum()))

# ---- Figure 4.1 : class distribution --------------------------------
counts = np.bincount(y)
pct = counts / counts.sum() * 100
fig, ax = new_fig('fig_4_1_class_distribution')
bars = ax.bar(['No Rain (0)', 'Rain (1)'], counts,
              color=['#1f77b4', '#2e7d32'], width=0.55)
for b, c, p in zip(bars, counts, pct):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + counts.max() * 0.015,
            f'{c:,}\n({p:.1f}%)', ha='center', va='bottom', fontsize=12)
ax.set_title('Class Distribution of the Target Variable (RainTomorrow)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_ylabel('Number of Records', fontsize=12, fontweight='bold')
ax.set_ylim(0, counts.max() * 1.18)
ax.grid(axis='y', linestyle=':', alpha=0.4)
ax.set_axisbelow(True)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
save_fig(fig, 'fig_4_1_class_distribution')


In [ ]:
# =====================================================================
# CELL 4 — Leakage-free pipeline + evaluation helper
#
# CHANGED FROM THE ORIGINAL: SimpleImputer and StandardScaler are now
# steps INSIDE the pipeline. Because the pipeline is fitted separately on
# each training fold, the median and the mean/standard deviation are
# computed from training data only. This is what Section 3.5 of the
# report claims, and it was not what the original code did.
#
# Order matters: impute -> scale -> SMOTE -> classify. SMOTE cannot run
# on data containing NaNs, and scaling before SMOTE keeps the synthetic
# samples in the same space as the real ones.
# =====================================================================
# The fitness function fits thousands of throwaway models whose only job is to
# RANK feature subsets, so it does not need the final model's convergence
# tolerance. Lowering max_iter here cuts the PSO cell several-fold and does not
# touch the reported results, which always use FINAL_MAX_ITER.
FINAL_MAX_ITER = 1000
FITNESS_MAX_ITER = 200


def build_pipeline(class_weight='balanced', max_iter=None):
    return ImbPipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
        ('smote',  SMOTE(random_state=SEED)),
        ('lr',     LogisticRegression(max_iter=max_iter or FINAL_MAX_ITER,
                                      class_weight=class_weight,
                                      solver='saga',
                                      random_state=SEED)),
    ])


def evaluate(X_in, y_in, label, class_weight='balanced'):
    """5-fold stratified CV. Returns metric means, stds, timings and the
    pooled confusion matrix across all five folds."""
    # Warm-up fit, discarded. The first fit in a process pays for BLAS thread
    # pool creation and memory allocation, which made fold 1 roughly twice as
    # slow as the rest. Without this, whichever system is evaluated first
    # absorbs that cost and looks slower than it is.
    warm_idx = np.random.default_rng(SEED).choice(
        len(y_in), size=min(5000, len(y_in)), replace=False)
    build_pipeline(class_weight).fit(X_in[warm_idx], y_in[warm_idx])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    m = {k: [] for k in ('acc', 'f1', 'prec', 'rec', 'auc')}
    fit_times, pred_times = [], []
    cm_total = np.zeros((2, 2), dtype=int)

    for train_idx, test_idx in skf.split(X_in, y_in):
        X_tr, X_te = X_in[train_idx], X_in[test_idx]
        y_tr, y_te = y_in[train_idx], y_in[test_idx]

        pipe = build_pipeline(class_weight)

        t0 = time.perf_counter()
        pipe.fit(X_tr, y_tr)
        fit_times.append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        y_pred = pipe.predict(X_te)
        pred_times.append(time.perf_counter() - t0)
        y_prob = pipe.predict_proba(X_te)[:, 1]

        m['acc'].append(accuracy_score(y_te, y_pred))
        m['f1'].append(f1_score(y_te, y_pred))
        m['prec'].append(precision_score(y_te, y_pred))
        m['rec'].append(recall_score(y_te, y_pred))
        m['auc'].append(roc_auc_score(y_te, y_prob))
        cm_total += confusion_matrix(y_te, y_pred)

    res = {k: (float(np.mean(v)), float(np.std(v))) for k, v in m.items()}
    res['train_time'] = (float(np.mean(fit_times)), float(np.std(fit_times)))
    res['pred_time'] = (float(np.mean(pred_times)), float(np.std(pred_times)))
    res['total_train_time'] = float(np.sum(fit_times))
    res['fold_train_times'] = [float(t) for t in fit_times]
    res['fold_pred_times'] = [float(t) for t in pred_times]
    res['median_train_time'] = float(np.median(fit_times))
    res['median_pred_time'] = float(np.median(pred_times))
    res['cm'] = cm_total.tolist()
    res['n_features'] = int(X_in.shape[1])

    print(f'\n=== {label} ({X_in.shape[1]} features) ===')
    for k, nm in [('acc', 'Accuracy'), ('f1', 'F1 Score'), ('prec', 'Precision'),
                  ('rec', 'Recall'), ('auc', 'AUC-ROC')]:
        print(f'  {nm:<10} {res[k][0]:.4f} ± {res[k][1]:.4f}')
    print(f'  Train time {res["train_time"][0]:.2f}s ± {res["train_time"][1]:.2f}'
          f'  (median {res["median_train_time"]:.2f}s, total {res["total_train_time"]:.2f}s)')
    print(f'  Per-fold   {[round(t, 2) for t in res["fold_train_times"]]}')
    spread = res['train_time'][1] / res['train_time'][0]
    if spread > 0.15:
        print(f'  !! training times vary by {spread*100:.0f}% across folds. Report the '
              f'median ({res["median_train_time"]:.2f}s) rather than the mean, and '
              f'close other applications before rerunning.')
    print(f'  Pred time  {res["pred_time"][0]:.4f}s ± {res["pred_time"][1]:.4f}')
    print('  Pooled confusion matrix (all 5 folds):')
    print('   ', cm_total.tolist())
    return res


In [ ]:
# =====================================================================
# CELL 5 — Logistic Regression system (all features)
# =====================================================================
res_lr = evaluate(X, y, 'Logistic Regression system')


In [ ]:
# =====================================================================
# CELL 6 — Binary PSO feature selection
#
# CHANGED FROM THE ORIGINAL:
#   * the subsample draw is now seeded, so the run is reproducible
#   * BinaryPSO is seeded via np.random.seed immediately before .optimize()
#   * the fitness pipeline imputes and scales inside each inner fold too
#   * c1/c2/w are set to values used in the BPSO literature rather than
#     the pyswarms documentation examples; the report must state whichever
#     values are actually used here
#
# STILL A KNOWN LIMITATION: selection runs once on a subsample of the full
# dataset, so the rows PSO sees overlap the test folds used in Cell 7.
# Fully nesting selection inside the outer CV would mean five PSO runs.
# If you do not do that, Chapter Five must disclose it plainly.
# =====================================================================
SAMPLE_SIZE = 20000
rng = np.random.default_rng(SEED)

idx_sample = []
for cls in (0, 1):
    cls_idx = np.where(y == cls)[0]
    n = int(SAMPLE_SIZE * len(cls_idx) / len(y))
    idx_sample.extend(rng.choice(cls_idx, n, replace=False))
idx_sample = np.array(idx_sample)

X_sample, y_sample = X[idx_sample], y[idx_sample]
print('PSO fitness sample size:', len(X_sample))
print('Sample class distribution:', np.bincount(y_sample))

MIN_FEATURES = 3
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)


def pso_fitness(particles):
    """particles: (n_particles, n_features) binary matrix -> (n_particles,) cost.
    Cost is 1 - mean F1 over a 3-fold CV of the selected subset."""
    costs = []
    for particle in particles:
        selected = np.where(particle == 1)[0]
        if len(selected) < MIN_FEATURES:
            costs.append(1.0)
            continue
        X_sel = X_sample[:, selected]
        scores = []
        for tr, te in inner_cv.split(X_sel, y_sample):
            pipe = build_pipeline(max_iter=FITNESS_MAX_ITER)
            pipe.fit(X_sel[tr], y_sample[tr])
            scores.append(f1_score(y_sample[te], pipe.predict(X_sel[te])))
        costs.append(1.0 - float(np.mean(scores)))
    return np.array(costs)


N_PARTICLES, N_ITERATIONS = 30, 100
N_FEATURES = X.shape[1]
options = {'c1': 2.0, 'c2': 2.0, 'w': 0.9, 'k': 5, 'p': 2}

print(f'\nRunning Binary PSO ({N_PARTICLES} particles, {N_ITERATIONS} iterations)...')
np.random.seed(SEED)                       # seeds pyswarms' swarm initialisation
t0 = time.perf_counter()
optimizer = ps.discrete.BinaryPSO(n_particles=N_PARTICLES,
                                  dimensions=N_FEATURES,
                                  options=options)
cost, best_pos = optimizer.optimize(pso_fitness, iters=N_ITERATIONS, verbose=True)
pso_search_time = time.perf_counter() - t0

cost_history = optimizer.cost_history
selected_features = np.where(best_pos == 1)[0]
selected_names = [feature_names[i] for i in selected_features]
dropped_names = [f for f in feature_names if f not in selected_names]

print(f'\nPSO search time: {pso_search_time:.1f}s')
print(f'Selected {len(selected_features)}/{N_FEATURES} features '
      f'({(1 - len(selected_features) / N_FEATURES) * 100:.1f}% reduction)')
print('Selected:', selected_names)
print('Dropped :', dropped_names)

# Checkpoint immediately: this cell is the expensive one, and losing its output
# to a disconnect or a crash means running it again from scratch.
np.save('cost_history.npy', np.array(cost_history))
np.save('best_pos.npy', best_pos)
json.dump({'selected_features': selected_names,
           'dropped_features': dropped_names,
           'selected_idx': selected_features.tolist(),
           'pso_search_time': pso_search_time,
           'pso_options': options,
           'final_cost': float(cost)},
          open('pso_checkpoint.json', 'w'), indent=2)
print('Checkpointed to pso_checkpoint.json / best_pos.npy / cost_history.npy')

# If you need to restart from here without re-running the search, use:
#   best_pos = np.load('best_pos.npy')
#   cost_history = np.load('cost_history.npy').tolist()
#   selected_features = np.where(best_pos == 1)[0]
#   selected_names = [feature_names[i] for i in selected_features]

# ---- Figure 4.3 : convergence curve ---------------------------------
# CHANGED: the y-axis is autoscaled to the data range. The original chart
# ran the axis from 0, which flattened a curve that only moves a few
# thousandths and made the optimisation look inert.
fig, ax = new_fig('fig_4_3_convergence')
ax.plot(range(1, len(cost_history) + 1), cost_history,
        color='#2e7d32', linewidth=2.0)
ax.set_title('PSO Convergence Curve (Iterations vs Best Cost)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax.set_ylabel('Best Cost (1 \u2212 F1 Score)', fontsize=12, fontweight='bold')
lo, hi = min(cost_history), max(cost_history)
pad = max((hi - lo) * 0.15, 1e-4)
ax.set_ylim(lo - pad, hi + pad)
bbox = dict(boxstyle='round,pad=0.35', fc='white', ec='#bbb', alpha=0.9)
ax.annotate(f'start {cost_history[0]:.4f}', xy=(1, cost_history[0]),
            xytext=(12, 14), textcoords='offset points', fontsize=10, bbox=bbox)
ax.annotate(f'final {cost_history[-1]:.4f}',
            xy=(len(cost_history), cost_history[-1]),
            xytext=(-95, 16), textcoords='offset points', fontsize=10, bbox=bbox)
ax.text(0.99, 0.94, f'total improvement: {cost_history[0] - cost_history[-1]:.4f}',
        transform=ax.transAxes, ha='right', fontsize=10, color='#555')
ax.grid(linestyle=':', alpha=0.45)
ax.set_axisbelow(True)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
save_fig(fig, 'fig_4_3_convergence')

# ---- Figure 4.4 : selected vs dropped features ----------------------
fig, ax = new_fig('fig_4_4_feature_selection')
mask = np.isin(np.arange(N_FEATURES), selected_features)
colors = ['#2e7d32' if m else '#c62828' for m in mask]
ax.bar(range(N_FEATURES), np.ones(N_FEATURES), color=colors, width=0.82)
ax.set_xticks(range(N_FEATURES))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
ax.set_yticks([])
ax.set_title('PSO Feature Selection: Selected vs Dropped Features',
             fontsize=13, fontweight='bold', pad=12)
handles = [plt.Rectangle((0, 0), 1, 1, color='#2e7d32'),
           plt.Rectangle((0, 0), 1, 1, color='#c62828')]
ax.legend(handles, [f'Selected ({mask.sum()})', f'Dropped ({(~mask).sum()})'],
          fontsize=10, loc='upper right')
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
save_fig(fig, 'fig_4_4_feature_selection')


In [ ]:
# =====================================================================
# CELL 7 — PSO-LR system (selected features only)
# =====================================================================
X_sel = X[:, selected_features]
res_pso = evaluate(X_sel, y, 'PSO-LR system')


In [ ]:
# =====================================================================
# CELL 8 — Confusion matrices (Figures 4.2 and 4.5)
#
# CHANGED: these are pooled across all five folds, so they are consistent
# with the metric means in Tables 4.2 and 4.4. The original figures came
# from a single fold, which is why their implied metrics never matched
# the tables.
# =====================================================================
def plot_cm(cm_list, title, name):
    a = np.array(cm_list, dtype=int)
    fig, ax = new_fig(name)
    im = ax.imshow(a, cmap='Blues')
    ax.set_title(title, fontsize=26, fontweight='bold', pad=22)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['No Rain', 'Rain'], fontsize=20)
    ax.set_yticklabels(['No Rain', 'Rain'], fontsize=20, rotation=90, va='center')
    ax.set_xlabel('Predicted', fontsize=24, fontweight='bold', labelpad=14)
    ax.set_ylabel('Actual', fontsize=24, fontweight='bold', labelpad=14)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{a[i, j]:,}', ha='center', va='center',
                    fontsize=30, fontweight='bold',
                    color='white' if a[i, j] > a.max() * 0.6 else '#333333')
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=17)
    for s in ax.spines.values():
        s.set_visible(False)
    fig.tight_layout()
    save_fig(fig, name)


plot_cm(res_lr['cm'], 'Confusion Matrix \u2014 Logistic Regression', 'fig_4_2_cm_lr')
plot_cm(res_pso['cm'], 'Confusion Matrix \u2014 PSO-LR', 'fig_4_5_cm_pso_lr')


In [ ]:
# =====================================================================
# CELL 9 — Comparison figures and the results table
# =====================================================================
metrics = ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'AUC-ROC']
keys = ['acc', 'f1', 'prec', 'rec', 'auc']
lr_vals = [res_lr[k][0] for k in keys]
pso_vals = [res_pso[k][0] for k in keys]

lr_lbl = f'Logistic Regression ({res_lr["n_features"]} features)'
pso_lbl = f'PSO-LR ({res_pso["n_features"]} features)'

# ---- Figure 4.6 : metric comparison ---------------------------------
fig, ax = new_fig('fig_4_6_performance')
x = np.arange(len(metrics)); w = 0.36
b1 = ax.bar(x - w / 2, lr_vals, w, label=lr_lbl, color='#1f77b4')
b2 = ax.bar(x + w / 2, pso_vals, w, label=pso_lbl, color='#2e7d32')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.012,
                f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=11)
ax.set_title('Performance Comparison: Logistic Regression vs PSO-LR',
             fontsize=15, fontweight='bold', pad=16)
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11.5, loc='upper right')
ax.grid(axis='y', linestyle=':', alpha=0.45)
ax.set_axisbelow(True)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
save_fig(fig, 'fig_4_6_performance')

# ---- Figure 4.7 : computation time ----------------------------------
# NEW FIGURE. This is the evidence the report currently lacks for its
# claim that the reduced system is more computationally efficient.
w_px, h_px = FIGSPEC['fig_4_7_computation_time']
fig, axes = plt.subplots(1, 2, figsize=(w_px / 100, h_px / 100), dpi=100)

panels = [
    (axes[0], 'Mean Training Time per Fold', 'Seconds',
     [res_lr['train_time'][0], res_pso['train_time'][0]],
     [res_lr['train_time'][1], res_pso['train_time'][1]], 1.0, '{:.3f}'),
    (axes[1], 'Mean Prediction Time per Fold', 'Milliseconds',
     [res_lr['pred_time'][0] * 1000, res_pso['pred_time'][0] * 1000],
     [res_lr['pred_time'][1] * 1000, res_pso['pred_time'][1] * 1000], 1.0, '{:.2f}'),
]

for ax, title, ylab, vals, errs, _s, fmt in panels:
    bars = ax.bar([0, 1], vals, 0.55, color=['#1f77b4', '#2e7d32'],
                  yerr=errs, capsize=6, error_kw={'ecolor': '#444', 'lw': 1.2})
    for b, v, e in zip(bars, vals, errs):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + e + max(vals) * 0.04,
                fmt.format(v), ha='center', va='bottom', fontsize=12,
                fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel(ylab, fontsize=12, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'Logistic Regression\n({res_lr["n_features"]} features)',
                        f'PSO-LR\n({res_pso["n_features"]} features)'], fontsize=11)
    ax.set_ylim(0, max(v + e for v, e in zip(vals, errs)) * 1.35)
    ax.grid(axis='y', linestyle=':', alpha=0.45)
    ax.set_axisbelow(True)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    pct = (vals[1] - vals[0]) / vals[0] * 100
    ax.text(0.99, 0.97, f'{pct:+.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=13, fontweight='bold',
            color='#2e7d32' if pct < 0 else '#c62828')

fig.suptitle('Computation Time: Logistic Regression vs PSO-LR',
             fontsize=15, fontweight='bold', y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_fig(fig, 'fig_4_7_computation_time')

# ---- Table 4.5 values, printed ready to type into the report --------
print('\n' + '=' * 78)
print('TABLE 4.5 — Performance Comparison of the Two Forecasting Systems')
print('=' * 78)
W = 36
print(f'{"Metric":<22}{lr_lbl:<{W}}{pso_lbl:<{W}}Difference')
print('-' * 78)
for nm, k in zip(metrics, keys):
    a, b = res_lr[k], res_pso[k]
    print(f'{nm:<22}{f"{a[0]:.4f} ± {a[1]:.4f}":<{W}}'
          f'{f"{b[0]:.4f} ± {b[1]:.4f}":<{W}}{b[0] - a[0]:+.4f}')
red = (1 - res_pso['n_features'] / res_lr['n_features']) * 100
print(f'{"Feature Count":<22}{res_lr["n_features"]:<{W}}{res_pso["n_features"]:<{W}}'
      f'{res_pso["n_features"] - res_lr["n_features"]:+d} ({red:.1f}% reduction)')
tt = (res_pso['train_time'][0] - res_lr['train_time'][0]) / res_lr['train_time'][0] * 100
pt = (res_pso['pred_time'][0] - res_lr['pred_time'][0]) / res_lr['pred_time'][0] * 100
mtt = ((res_pso['median_train_time'] - res_lr['median_train_time'])
       / res_lr['median_train_time'] * 100)
print(f'{"Mean train time (s)":<22}{res_lr["train_time"][0]:<{W}.3f}'
      f'{res_pso["train_time"][0]:<{W}.3f}{tt:+.1f}%')
print(f'{"Mean predict time (s)":<22}{res_lr["pred_time"][0]:<{W}.4f}'
      f'{res_pso["pred_time"][0]:<{W}.4f}{pt:+.1f}%')
print(f'{"Median train time (s)":<22}{res_lr["median_train_time"]:<{W}.3f}'
      f'{res_pso["median_train_time"]:<{W}.3f}{mtt:+.1f}%')
print(f'\nPSO search cost (one-off): {pso_search_time:.1f}s')
saved = res_lr['median_train_time'] - res_pso['median_train_time']
if saved > 0:
    print(f'Break-even: {pso_search_time / saved:,.0f} training runs before the '
          f'search pays for itself ({saved:.2f}s saved per run)')
else:
    print(f'No training-time saving measured ({saved:+.2f}s per run) - the '
          f'reduction cannot be reported as a computational benefit.')
print('=' * 78)

# The run configuration is saved WITH the results. Without this, a rerun at
# different settings produces numbers that cannot be traced back to the
# conditions that made them - which is exactly how a 30-iteration result ends
# up sitting under a methodology section that says one hundred.
json.dump({'lr': res_lr, 'pso_lr': res_pso,
           'selected_features': selected_names,
           'dropped_features': dropped_names,
           'pso_search_time': pso_search_time,
           'config': {
               'seed': SEED,
               'n_particles': N_PARTICLES,
               'n_iterations': N_ITERATIONS,
               'iterations_actually_run': len(cost_history),
               'pso_options': options,
               'sample_size': SAMPLE_SIZE,
               'min_features': MIN_FEATURES,
               'fitness_max_iter': FITNESS_MAX_ITER,
               'final_max_iter': FINAL_MAX_ITER,
               'cv_folds': 5,
               'inner_cv_folds': 3,
               'solver': 'saga',
               'n_records': int(len(y)),
               'n_features_total': int(N_FEATURES),
           }},
          open('results.json', 'w'), indent=2)

if len(cost_history) != N_ITERATIONS:
    print(f'\n!! WARNING: N_ITERATIONS is {N_ITERATIONS} but the search recorded '
          f'{len(cost_history)} iterations. Chapter Three must state '
          f'{len(cost_history)}, or rerun Cell 6.')
print('\nAll figures written to ./figures/ ; numbers saved to results.json')


In [ ]:
# =====================================================================
# CELL 10 — Ablation: is class_weight='balanced' doing anything?
#
# Inside an ImbPipeline, SMOTE resamples the training fold BEFORE the
# classifier is fitted. By the time LogisticRegression computes its
# 'balanced' weights, the classes are already equal, so both weights come
# out at 1.0 and the setting is very close to a no-op.
#
# Run this to confirm it on the real data. If the two sets of numbers are
# effectively identical, Section 3.6 should not describe class weighting
# as "a secondary mechanism" for handling imbalance - it is redundant,
# and SMOTE alone is what moves precision and recall.
# =====================================================================
res_lr_nocw = evaluate(X, y, 'Logistic Regression (SMOTE only, no class weight)',
                       class_weight=None)
print(f"\nPrecision  SMOTE + class_weight : {res_lr['prec'][0]:.4f}")
print(f"Precision  SMOTE only           : {res_lr_nocw['prec'][0]:.4f}")
print(f"Recall     SMOTE + class_weight : {res_lr['rec'][0]:.4f}")
print(f"Recall     SMOTE only           : {res_lr_nocw['rec'][0]:.4f}")
delta = abs(res_lr['prec'][0] - res_lr_nocw['prec'][0])
verdict = ('no-op' if delta < 0.005 else 'material')
print(f'\nclass_weight is {verdict} here (precision delta {delta:.4f}).')

# append to results.json so the ablation travels with the rest of the numbers
_r = json.load(open('results.json'))
_r['ablation_smote_only'] = res_lr_nocw
_r['ablation_verdict'] = verdict
json.dump(_r, open('results.json', 'w'), indent=2)
print('Ablation appended to results.json')
